# Interpolation and Galerkin approximation

Why does the finite element solution converge faster in $L^2$ than in the energy norm? We compare nodal interpolation and the Galerkin solution for the same Poisson problem, then refine the mesh.

In [ ]:
from ngsolve import *
from netgen.occ import unit_square
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt

## 1. Problem setup

On $\Omega=(0,1)^2$, solve

$$
\begin{align*}
-\Delta u=f \quad\text{in }\Omega,\qquad u=0 \quad\text{on }\partial\Omega.
\end{align*}
$$

We choose

$$
\begin{align*}
u_{\rm ex}=x(1-x)y(1-y),\qquad f=2\bigl(x(1-x)+y(1-y)\bigr).
\end{align*}
$$

For a conforming triangular mesh $\mathcal T_h$, let $V_h=S_1^0(\mathcal T_h)\cap H_0^1(\Omega)$. The Galerkin solution satisfies

$$
\begin{align*}
\mathcal A(u_h,v_h)=\ell(v_h)\quad\forall v_h\in V_h,\qquad \mathcal A(w,v)=\int_\Omega\nabla w\cdot\nabla v\,dx,\qquad \ell(v)=\int_\Omega fv\,dx.
\end{align*}
$$

The energy norm is $\|v\|_{\mathcal A}=|v|_{H^1}=\|\nabla v\|_{L^2}$.

In [ ]:
u_ex = x*(1-x)*y*(1-y)
grad_u_ex = CF((u_ex.Diff(x), u_ex.Diff(y)))
f = 2*(x*(1-x) + y*(1-y))

initial_maxh = 0.35
num_levels = 5

mesh = Mesh(unit_square.GenerateMesh(maxh=initial_maxh))
Draw(u_ex, mesh, name="Exact solution", deformation=True);

## 2. Two functions in the same finite element space

The nodal interpolant is defined by $(I_hu)(x_i)=u(x_i)$ at every mesh vertex. For $P_1$ elements, its coefficients are exactly these vertex values. In this space, `Set(exact, dual=True)` performs nodal interpolation.

The Galerkin coefficients instead solve the linear system obtained from $\mathcal A$ and $\ell$. The homogeneous boundary coefficients remain zero in both constructions.

In [ ]:
def galerkin_solution(fes):
    u, v = fes.TnT()
    a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
    l = LinearForm(f*v*dx).Assemble()
    uh = GridFunction(fes)
    uh.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * l.vec
    return uh


fes = H1(mesh, order=1, dirichlet=".*")
Ihu = GridFunction(fes)
Ihu.Set(u_ex, dual=True)
uh = galerkin_solution(fes)

Draw(Ihu, mesh, name="Nodal interpolant", deformation=True)
Draw(uh, mesh, name="Galerkin solution", deformation=True)
Draw(Norm(Ihu-uh), mesh, name="Interpolant minus Galerkin solution", deformation=True);

### Exact nodal values or smallest energy error?

Galerkin orthogonality states that $\mathcal A(u-u_h,v_h)=0$ for every $v_h\in V_h$. Taking $v_h=u_h-I_hu$ gives the Pythagorean identity

$$
|u-I_hu|_{H^1}^2
=
|u-u_h|_{H^1}^2+|u_h-I_hu|_{H^1}^2.
$$

Thus $u_h$ has no larger energy error than $I_hu$, even though it need not reproduce the exact nodal values. This best-approximation statement concerns the **energy norm**, but it does not assert that $u_h$ minimizes the $L^2$ error.

In [ ]:
def errors(mesh, approximation):
    difference = u_ex-approximation
    gradient_difference = grad_u_ex-grad(approximation)
    L2_error = sqrt(Integrate(difference**2, mesh))
    energy_error = sqrt(Integrate(gradient_difference**2, mesh))
    return L2_error, energy_error


L2_I, energy_I = errors(mesh, Ihu)
L2_G, energy_G = errors(mesh, uh)
discrete_gradient = grad(uh)-grad(Ihu)
distance_squared = Integrate(discrete_gradient*discrete_gradient, mesh)

print(f"Interpolation: L2 error = {L2_I:.6e}, energy error = {energy_I:.6e}")
print(f"Galerkin:      L2 error = {L2_G:.6e}, energy error = {energy_G:.6e}")
print(f"Pythagoras: {energy_I**2:.10e} = {energy_G**2:.10e} + {distance_squared:.10e}")

## 3. Refinement and error rates

We solve on successively refined meshes. For the convergence plots, we use the mesh-size measure

$$
\begin{align*}
h=\max_{T\in\mathcal T_h}h_T,\qquad h_T=\sqrt{2|T|}.
\end{align*}
$$

On these affine triangles, `specialcf.mesh_size` equals $\sqrt{2|T|}$ and is constant on each element. We interpolate it into the piecewise constant space `L2(mesh, order=0)` and take the largest coefficient. On this shape-regular mesh family, this measure is equivalent to the largest element diameter and gives the same convergence orders. Both measures halve under each uniform refinement. The mesh generator's parameter `maxh` is only used for the initial mesh.

For the smooth solution on this convex square, the theory predicts

$$
\begin{align*}
|u-I_hu|_{H^1},\ |u-u_h|_{H^1}=O(h), \qquad \|u-I_hu\|_{L^2},\ \|u-u_h\|_{L^2}=O(h^2).
\end{align*}
$$

Interpolation supplies the approximation bound. For the Galerkin $L^2$ error, the extra factor $h$ follows from Aubin–Nitsche and the $H^2$ regularity of the dual problem.

In [ ]:
def mesh_size(mesh):
    h = GridFunction(L2(mesh, order=0))
    Draw(specialcf.mesh_size, mesh, "mesh size")
    h.Interpolate(specialcf.mesh_size)
    return h.vec.FV().NumPy().max()


mesh = Mesh(unit_square.GenerateMesh(maxh=initial_maxh))
mesh.Refine()
hs, ndofs = [], []
L2_interpolation, energy_interpolation = [], []
L2_galerkin, energy_galerkin = [], []

for level in range(num_levels):
    fes = H1(mesh, order=1, dirichlet=".*")
    Ihu = GridFunction(fes)
    Ihu.Set(u_ex, dual=True)
    uh = galerkin_solution(fes)
    L2_I, energy_I = errors(mesh, Ihu)
    L2_G, energy_G = errors(mesh, uh)

    hs.append(mesh_size(mesh))
    ndofs.append(fes.ndof)
    L2_interpolation.append(L2_I)
    energy_interpolation.append(energy_I)
    L2_galerkin.append(L2_G)
    energy_galerkin.append(energy_G)

    if level < num_levels-1:
        mesh.Refine()

hs = np.array(hs)
print("level    dofs          h       energy error     L2 error")
for level, (ndof, h, e1, e0) in enumerate(
        zip(ndofs, hs, energy_galerkin, L2_galerkin)):
    print(f"{level:3d}  {ndof:7d}  {h:10.4e}  {e1:14.6e}  {e0:12.6e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, interpolation_errors, galerkin_errors, power, title in zip(
        axes, [energy_interpolation, L2_interpolation],
        [energy_galerkin, L2_galerkin], [1, 2],
        ["Energy error", "L2 error"]):
    ax.loglog(hs, interpolation_errors, "o-", label="nodal interpolation")
    ax.loglog(hs, galerkin_errors, "s-", label="Galerkin solution")
    reference = galerkin_errors[-1]*(hs/hs[-1])**power
    ax.loglog(hs, reference, "k--", label=rf"reference $h^{power}$")
    ax.set(xlabel=r"$h=\max_T\sqrt{2|T|}$", ylabel="error", title=title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.show()

The experimental order of convergence (EOC) between two consecutive meshes is

$$
\begin{align*}
\mathrm{EOC}_i=\frac{\log(e_i/e_{i+1})}{\log(h_i/h_{i+1})}.
\end{align*}
$$

A rate close to $1$ means that halving $h$ approximately halves the error, while a rate close to $2$ means that it approximately quarters the error.

In [ ]:
print("                  interpolation             Galerkin")
print("mesh pair       energy       L2         energy       L2")
for i in range(len(hs)-1):
    denominator = np.log(hs[i]/hs[i+1])
    rates = [np.log(values[i]/values[i+1])/denominator
             for values in [energy_interpolation, L2_interpolation, energy_galerkin, L2_galerkin]]
    print(f"{i:2d} -> {i+1:<2d}     " + "    ".join(f"{rate:8.3f}" for rate in rates))

## Observe

- Why can the interpolant have exact values at every vertex and still have a larger energy error?
- Which part of the argument gives order $h$ in the energy norm? Where does the second factor $h$ in the Galerkin $L^2$ estimate enter?
- The mesh and both numerical solutions may already look accurate on screen. Which plot reveals the difference between the two error norms?

**Takeaway.** Nodal interpolation provides a comparison function. Galerkin orthogonality gives best approximation in energy, while dual regularity gives the faster $L^2$ rate.

[Next: shape regularity →](02_shape_regularity.ipynb)

[Lecture notebooks](../README.md) · [Lecture notes](../../script/README.md) · [Course overview](../../index.ipynb)